# 2 — Préparation des données (CRISP-DM Phase 3)

Ce notebook est la **source canonique** pour la préparation des données du projet Inved Corp. Il :

1. Charge les données (`./data/train.csv`).
2. Élimine les outliers et applique la transformation logarithmique sur la cible.
3. Sépare le train et le test (split déterministe).
4. Construit **trois variantes de préprocesseur**, chacune justifiée selon les hypothèses des familles de modèles qui les consomment.
5. Définit des fonctions utilitaires partagées (`rmsle_score`, `predicted_vs_actual_plot`, `publish_result`).

Chaque notebook de modélisation (3a, 3b, 3c, 3d) démarre par :

```python
%load_ext autoreload
%autoreload 2
%run 2_data_prep.ipynb
```

…ce qui ré-exécute ce notebook dans son propre kernel et expose toutes les variables, transformateurs et fonctions utilitaires sans import ad-hoc.

## 3.1 Chargement des données et imports

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('./data/train.csv', sep=',')
print(f"Dimensions brutes : {df.shape}")

## 3.2 Suppression des observations atypiques

Sur la base de l'analyse exploratoire (NB1) et du papier de référence **De Cock (2011)**, nous retirons uniquement les rares maisons dont la surface habitable (`GrLivArea > 4000`) **et** le prix de vente (`SalePrice < 300 000`) trahissent une vente forcée ou un défaut majeur — pas les maisons de luxe à très grande surface, qu'on veut garder pour que le modèle apprenne le régime haut-de-gamme.

In [ ]:
outliers_index = df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300_000)].index
print(f"Nombre de points supprimés : {len(outliers_index)}")

df_cleaned = df.drop(outliers_index).copy()
print(f"Dimensions après suppression : {df_cleaned.shape}")

## 3.3 Transformation logarithmique de la cible

La variable `SalePrice` est très asymétrique à droite. On applique `log1p` pour stabiliser la distribution. La métrique du projet est la **RMSLE** (RMSE sur `log1p(SalePrice)`) — une erreur de 10 k$ sur une maison à 100 k$ ne se mesure pas comme une erreur de 10 k$ sur une villa à 1 M$. La RMSLE pénalise *proportionnellement*, ce qui correspond au sens métier de Inved Corp.

In [ ]:
df_cleaned['SalePrice_log'] = np.log1p(df_cleaned['SalePrice'])

## 3.4 Découpage train/test

Le split est fait **avant** toute transformation (imputation, encodage, mise à l'échelle) afin de garantir qu'aucune statistique calculée sur le test ne fuite dans l'entraînement. Toutes les transformations seront ajustées **uniquement** sur `X_train` via les `Pipeline` / `ColumnTransformer` de la section 3.5.

Graine fixée à `42` pour la reproductibilité : chaque notebook de modélisation qui ré-exécute ce fichier obtiendra le même `X_train` / `X_test`.

In [ ]:
X = df_cleaned.drop(columns=['SalePrice', 'SalePrice_log', 'Id'], errors='ignore')
y_log = df_cleaned['SalePrice_log']

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=RANDOM_STATE
)

print(f"X_train : {X_train.shape}   |   X_test : {X_test.shape}")
print(f"NaN dans X_train : {int(X_train.isna().sum().sum())} cellules sur {X_train.size}")

## 3.5 Trois variantes de préprocesseur, une par famille de modèles

Chaque famille de modèles a des **hypothèses différentes sur les données d'entrée** ; il serait erroné d'appliquer le même prétraitement à tous. Nous construisons trois variantes nommées, et chaque notebook de modélisation choisira la sienne en fonction des hypothèses des modèles qu'il contient.

| Variante                  | Imputation             | Encodage catégoriel                                                | Mise à l'échelle | Famille cible                                  |
|---------------------------|------------------------|---------------------------------------------------------------------|------------------|-----------------------------------------------|
| `preprocessor_scaled`     | médiane / 0 / mode / `'None'` | Ordinal (qualités) + OneHot (nominal)                       | **StandardScaler** | Linéaires, KNN, MLP — sensibles à l'échelle    |
| `preprocessor_encoded`    | idem                   | idem                                                                | aucune           | Arbres sklearn (DT, RF, GB, AdaBoost)         |
| `preprocessor_native`     | **aucune** (NaN gérés par le modèle) | aucune (`pd.Categorical` brut)                          | aucune           | XGBoost, LightGBM, CatBoost — gestion native  |

Les justifications détaillées (assumptions du modèle → choix du préprocesseur) sont rappelées dans chaque notebook de modélisation au début de chaque modèle (bloc « Hypothèses du modèle »).

### 3.5.1 Groupes de colonnes

On définit les listes de colonnes à partir de `X_train`, en distinguant :

- **Numériques** où NA = `0` (absence d'équipement, ex. `GarageArea` quand pas de garage) → imputation constante par `0`.
- **Numériques** où NA = vraie valeur manquante (ex. `LotFrontage`) → imputation par la médiane.
- **Ordinales** : échelles de qualité (`Ex > Gd > TA > Fa > Po > None`) → `OrdinalEncoder` avec ordre explicite.
- **Nominales** où NA = `'None'` (absence) → imputation constante puis encodage.
- **Nominales** où NA = vraie valeur manquante → imputation par le mode puis encodage.
- `Utilities` est quasi-constante (1457/1458 valent `AllPub`) et donc retirée.

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()

num_zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1',
    'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath',
    'BsmtHalfBath', 'MasVnrArea',
]
num_median_cols = [c for c in num_cols if c not in num_zero_cols]

ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
    'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC',
]
ordinal_cats = [['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']] * len(ordinal_cols)

all_cat_cols = X_train.select_dtypes(include=['object', 'string']).columns.tolist()
nominal_cols = [c for c in all_cat_cols if c not in ordinal_cols]

nominal_none_cols = [
    'Alley', 'BsmtFinType1', 'BsmtFinType2', 'BsmtExposure',
    'GarageType', 'GarageFinish', 'Fence', 'MiscFeature', 'MasVnrType',
]
nominal_mode_cols = [c for c in nominal_cols if c not in nominal_none_cols and c != 'Utilities']

print(f"  num_zero        ({len(num_zero_cols):2d} cols)")
print(f"  num_median      ({len(num_median_cols):2d} cols)")
print(f"  ordinal         ({len(ordinal_cols):2d} cols)")
print(f"  nominal_none    ({len(nominal_none_cols):2d} cols)")
print(f"  nominal_mode    ({len(nominal_mode_cols):2d} cols)")
print(f"  total           ({len(num_zero_cols)+len(num_median_cols)+len(ordinal_cols)+len(nominal_none_cols)+len(nominal_mode_cols):2d} cols routées, sur {X_train.shape[1]} dans X_train ; 'Utilities' droppée)")

### 3.5.2 `preprocessor_scaled` — pour les modèles sensibles à l'échelle

**Consommé par :** OLS, Ridge, Lasso, ElasticNet, KNN, MLPRegressor.

**Pourquoi :** ces modèles reposent soit sur une *distance euclidienne* (KNN), soit sur une *pénalité de norme* (Ridge/Lasso/ElasticNet — la régularisation suppose que toutes les variables sont sur une échelle comparable, sinon les grandes valeurs absorbent toute la pénalité), soit sur une *descente de gradient* (MLP — converge mal sur des features non normalisées). OLS sans pénalité n'a pas besoin du scaler en théorie mais on l'inclut pour cohérence et pour stabiliser la condition numérique.

In [ ]:
def _build_column_transformer(scale: bool) -> ColumnTransformer:
    num_zero_steps = [('imputer', SimpleImputer(strategy='constant', fill_value=0))]
    num_median_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        num_zero_steps.append(('scaler', StandardScaler()))
        num_median_steps.append(('scaler', StandardScaler()))

    return ColumnTransformer(
        transformers=[
            ('num_zero',   Pipeline(num_zero_steps),                                                                  num_zero_cols),
            ('num_median', Pipeline(num_median_steps),                                                                num_median_cols),
            ('ord',        Pipeline([
                              ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
                              ('encoder', OrdinalEncoder(categories=ordinal_cats, handle_unknown='use_encoded_value', unknown_value=-1)),
                          ]), ordinal_cols),
            ('nom_none',   Pipeline([
                              ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
                              ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                          ]), nominal_none_cols),
            ('nom_mode',   Pipeline([
                              ('imputer', SimpleImputer(strategy='most_frequent')),
                              ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                          ]), nominal_mode_cols),
        ],
        remainder='drop',
    )

preprocessor_scaled = _build_column_transformer(scale=True)

### 3.5.3 `preprocessor_encoded` — pour les arbres sklearn

**Consommé par :** DecisionTreeRegressor, RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor.

**Pourquoi :** un arbre de décision **partitionne l'espace par seuils** (ex. `GrLivArea > 1500`). Cette opération est **invariante par toute transformation monotone croissante** : standardiser une variable ne change ni l'ordre des observations ni les seuils possibles. Le `StandardScaler` est donc une opération *inutile* (sans dégrader la performance, mais inutile). On garde imputation et encodage car les arbres sklearn n'acceptent pas les NaN ni les chaînes brutes.

In [ ]:
preprocessor_encoded = _build_column_transformer(scale=False)

### 3.5.4 `preprocessor_native` — pour les boosters modernes

**Consommé par :** XGBoost, LightGBM, CatBoost.

**Pourquoi :** ces bibliothèques implémentent leurs propres mécanismes pour les valeurs manquantes (`NaN`-aware splits) et pour les variables catégorielles (`enable_categorical=True` pour XGBoost ≥ 1.6, `categorical_feature` pour LightGBM, `cat_features` pour CatBoost). Imputer ou encoder en amont **gaspille de l'information** : le modèle perd la capacité de traiter NaN comme un signal en soi, et l'encodage OneHot fait exploser la dimensionnalité (79 → 256 colonnes dans notre cas). Le préprocesseur natif se contente donc de convertir les colonnes `object` en `pd.Categorical` en mémorisant les catégories observées à l'entraînement, ce qui garantit la cohérence train/test.

In [ ]:
class NativeCategoricalTransformer(BaseEstimator, TransformerMixin):
    """Coerce object/string columns to pd.Categorical with train-fixed categories.

    Numerical columns and NaN cells pass through untouched — the consumer
    model (XGBoost/LightGBM/CatBoost) handles them natively. Test-set values
    not seen at fit time are coerced to NaN (consumer model handles them too).
    """

    def fit(self, X, y=None):
        self.cat_cols_ = X.select_dtypes(include=['object', 'string']).columns.tolist()
        self.categories_ = {
            col: X[col].astype('category').cat.categories
            for col in self.cat_cols_
        }
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cat_cols_:
            known = self.categories_[col]
            # Coerce unknown values to NaN before building the Categorical,
            # to avoid pandas4 deprecation warning + match consumer expectation.
            col_vals = X[col].where(X[col].isin(known))
            X[col] = pd.Categorical(col_vals, categories=known)
        return X


preprocessor_native = NativeCategoricalTransformer()


## 3.6 Sentinelle anti-fuite (T06)

Le préprocesseur est désormais la *source unique de vérité* pour le nettoyage. Pour empêcher qu'un futur contributeur ne réintroduise un nettoyage en amont (e.g. un `fillna()` global), on ajoute une assertion explicite : `X_train` **doit** contenir des NaN à ce stade. Si quelqu'un imp à un endroit en amont, l'assertion casse et la dérive est rattrapée.

In [ ]:
assert X_train.isna().sum().sum() > 0, (
    "T06 sentinel — Pipeline must receive raw data with NAs to impute. "
    "If this assertion fails, someone reintroduced upstream cleaning."
)

## 3.7 Fonctions utilitaires partagées

Ces fonctions sont importées (via `%run`) par tous les notebooks de modélisation et par le notebook d'évaluation. Elles définissent :

- `rmsle_score(y_true_log, y_pred_log)` — la métrique du projet (RMSE en espace log = RMSLE en espace brut).
- `cv_rmsle(pipe, X, y_log, cv=5)` — RMSLE par cross-validation 5-folds.
- `predicted_vs_actual_plot(...)` — le scatter exigé par la consigne (« chaque modèle doit avoir un scatter predicted-vs-actual »).
- `publish_result(...)` — sérialise les résultats d'un modèle dans `results/family_<name>.json` (lu par `4_evaluation.ipynb`).

In [ ]:
def rmsle_score(y_true_log, y_pred_log) -> float:
    """RMSE sur la cible log-transformée = RMSLE sur les prix bruts."""
    return float(np.sqrt(mean_squared_error(y_true_log, y_pred_log)))


def cv_rmsle(pipe, X, y_log, cv: int = 5) -> float:
    """RMSLE moyenne en cross-validation k-fold (scoring scikit-learn `neg_root_mean_squared_error`)."""
    scores = cross_val_score(
        pipe, X, y_log, cv=cv,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
    )
    return float(-scores.mean())


def predicted_vs_actual_plot(y_true_log, y_pred_log, title: str = 'Predicted vs Actual', ax=None):
    """Scatter predicted-vs-actual avec diagonale parfaite en pointillé rouge."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true_log, y_pred_log, alpha=0.4, s=20)
    lo = float(min(np.min(y_true_log), np.min(y_pred_log)))
    hi = float(max(np.max(y_true_log), np.max(y_pred_log)))
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='y = x')
    ax.set_xlabel('Prix réel (log)')
    ax.set_ylabel('Prix prédit (log)')
    ax.set_title(title)
    ax.legend(loc='upper left')
    return ax


RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)


def publish_result(family: str, model: str,
                   cv_rmsle: float = None,
                   holdout_rmsle: float = None,
                   fit_time_s: float = None,
                   params: dict = None,
                   notes: str = None) -> Path:
    """Append/upsert (par `model` name) un résultat dans results/family_<family>.json."""
    path = RESULTS_DIR / f'family_{family}.json'
    data = json.loads(path.read_text()) if path.exists() else []
    data = [r for r in data if r.get('model') != model]
    data.append({
        'family': family,
        'model': model,
        'cv_rmsle': cv_rmsle,
        'holdout_rmsle': holdout_rmsle,
        'fit_time_s': fit_time_s,
        'params': params,
        'notes': notes,
    })
    path.write_text(json.dumps(data, indent=2, default=str))
    return path

## 3.8 Vérification finale

Vérification rapide que les trois préprocesseurs et tous les helpers sont définis. **On ne `fit()` rien ici** — chaque notebook de modélisation `fit`tera son propre `Pipeline(preprocessor_X, model)` une seule fois.

In [ ]:
print("Variantes de préprocesseur :")
print(f"  preprocessor_scaled  : {type(preprocessor_scaled).__name__}")
print(f"  preprocessor_encoded : {type(preprocessor_encoded).__name__}")
print(f"  preprocessor_native  : {type(preprocessor_native).__name__}")
print()
print("Helpers : rmsle_score, cv_rmsle, predicted_vs_actual_plot, publish_result")
print()
print(f"Données prêtes — X_train {X_train.shape}, X_test {X_test.shape}")